#Tokenizacja BPE
**Tokenizator** dzieli wejściowy tekst na krótsze jednostki leksykalne (**tokeny**). Każdy token ma swój unikalny numeryczny identyfikator. W metodzie BPE **słownik tokenów** zawiera wszystkie pojedyncze znaki, najczęściej występujące sekwencje znaków (pary, trójki, ...) jak i najczęściej występujące słowa. Pozwala to na na odwracalną tokenizację każdego tekstu, nawet zawierającego słowa w innym języku bądź błędnie zapisane. W modelach GPT-3.5 and GPT-4 słownik tokenów liczy ponad 100 tys. elementów, a zakodowanie jednego słowa w języku angielskim wymaga średnio 1,3 tokena.

Słownik zawiera również **tokeny specjalne** wykorzystywane podczas uczenia modelu i inferencji. W zależnoności od modelu językowego wykorzystywane są różne tokeny specjalne, np.:
- `<bos>, <s>` - beginning of sequence
- `<eos>, </s>` - end of sequence
- `<pad>` - padding token
- `<mask>` - mask token  

W notatniku wykorzystamy wytrenowany tokenizator `polish-gpt2-small` klasy BPE (*Byte-Pair Encoding*) używany w polskiej wersji modelu językowego GPT-2 z serwisu HuggingFace.

Więcej informacji na temat tokenizatorów w bibliotece HuggingFace dostępnych jest [tutaj](https://huggingface.co/docs/transformers/v4.57.1/main_classes/tokenizer).

In [ ]:
import numpy as np

import torch
import torch.nn as nn
from torch import Tensor

# Moduły biblioteki HuggingFace
import transformers

print(f"Wersja PyTorch: {torch.__version__}")
print(f"Wersja HuggingFace transformers: {transformers.__version__}")

In [ ]:
from transformers import AutoTokenizer

model_name = "sdadas/polish-gpt2-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
print(tokenizer)

In [ ]:
isinstance(tokenizer, transformers.PreTrainedTokenizerFast)


**Przydatne atrybuty** klasy `PreTrainedTokenizerFast`:
*   `vocab_size` - rozmiar słownika

**Przydatne metody** klasy `Tokenizer`:

*   `tokenize` - dzieli tekst na tokeny i zwraca listę tekstowych tokenów ([link](https://huggingface.co/docs/transformers/v4.44.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizer.tokenize))
*   `convert_tokens_to_ids` - zamienia token lub listę tokenów na listę identyfikatorów tokenów [link](https://huggingface.co/docs/transformers/v4.44.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizerFast.convert_tokens_to_ids)
*   `convert_ids_to_tokens` - zamienia identyfikator tokenu lub listę identyfikatorów tokenów na listę tokenów [link](https://huggingface.co/docs/transformers/v4.44.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizerFast.convert_ids_to_tokens)
*   `encode` - dzieli tekst na tokeny i zwraca listę numerycznych identyfikatorów tokenów ([link](https://huggingface.co/docs/transformers/v4.44.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizer.encode))
*   `decode` - rekonstruuje tekst z listy numerycznych identyfikatorów tokenów ([link](https://huggingface.co/docs/transformers/v4.44.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode))

In [ ]:
txt = "Litwo, Ojczyzno moja! ty jesteś jak zdrowie; Ile cię trzeba cenić,"

In [ ]:
print(txt)
print()

tokenized_txt = tokenizer.tokenize(txt)
print("Tekst po tokenizacji:")
print(tokenizer.tokenize(txt))
print()

print("Tekst po zamianie na identyfikatory tokenów:")
encoded_txt = tokenizer.convert_tokens_to_ids(tokenized_txt)
print(encoded_txt)
print()

print("Zrekonstruowany tekst:")
decoded_txt = tokenizer.decode(encoded_txt)
print(decoded_txt)
print()


In [ ]:
txt = "xsdfj sdfkj sfewjh!@#"

tokenized_txt = tokenizer.tokenize(txt)
print("Tekst po tokenizacji:")
print(tokenized_txt)
print()

print("Tekst po zamianie na identyfikatory tokenów:")
encoded_txt = tokenizer.convert_tokens_to_ids(tokenized_txt)
print(encoded_txt)
print()

print("Zrekonstruowany tekst:")
decoded_txt = tokenizer.decode(encoded_txt)
print(decoded_txt)
print()

Domyślna metoda tokenizera (`__call_`) (patrz: [link](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__)) zamienia listę łańcuchów tekstowych na **wsad treningowy złożony z sekwencji identyfikatorów tokenów dopełnionych do równej długości** tokenem `<pad>`. Wsad jest obiektem klasy `BatchEncoding` (patrz: [link](https://huggingface.co/docs/transformers/v4.44.0/en/main_classes/tokenizer#transformers.BatchEncoding)).

In [ ]:
# Tokenizacja tekstu i utworzenie wsadu treningowego
texts = ["Ala ma kota", "W sławnym mieście Pacanowie"]
encoded_inputs = tokenizer(texts, return_tensors='pt', padding=True)
print(type(encoded_inputs))

input_ids = encoded_inputs['input_ids']
attention_mask = encoded_inputs['attention_mask']

# Print the batch
print(input_ids)
print(attention_mask)

###Specyfika tokenizatora BPE

Wynik tokenizacji tego samego słowa jest różny, w zalezności czy rozpoczyna się dużą czy małą literą oraz czy jest poprzedzone spacją.

In [ ]:
print(f"{tokenizer('Słońce')=}")
print(f"{tokenizer(' Słońce')=}")
print(f"{tokenizer('słońce')=}")
print(f"{tokenizer(' słońce')=}")

Kodowanie liczb nie rządzi się logicznymi regułami. Zbitki cyfr występujących często w zbiorze treningowym tokenizatora kodowane są jako pojedynczy token.

In [ ]:
tokens = tokenizer("56873+3184623=123456789-1000000000")['input_ids']
for token in tokens:
    print(f"tokend id: {token:5d} -> {tokenizer.decode(token)}")


Wyświetlenie kilku losowo wybranych tokenów ze słownika.

In [ ]:
import random
print(f"Rozmiar słownika: {tokenizer.vocab_size} tokenów")

selected_token_ids = random.sample(range(tokenizer.vocab_size), k=20)
for token_id in selected_token_ids:
    print(f"id: {token_id} -> {tokenizer.decode(token_id)}")

# Uruchomienie modelu językowego w loklanym środowisku HuggingFace

Upewnij się, że notatnik jest uruchomiony na maszynie z GPU. Jeśli nie, zmień typ środowiska (menu: Runtime | Change runtime type).

In [ ]:
!nvidia-smi

Pobranie wag i uruchomienie polskiego modelu językowego **Bielik-1.5B-v3**.
Model  wytrenowano dostrajając wstępnie wytrenowany model Qwen2.5 1.5B na polskich zasobach językowych (303 miliony dokumentów zawierające 292 miliardy tokenów). Udostępniono zarówno model bazowy (statystyczny model języka), jak i model dostrojony do wykonywania poleceń i wykorzystania w systemach dialogowych (wersja Instruct).

W dalszej części notatnika użyjemy model bazowy **Bielik-1.5B-v3**, NIE dostrojony do wykonywania poleceń.
Więcej informacji: [link](https://huggingface.co/speakleash/Bielik-1.5B-v3).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "speakleash/Bielik-1.5B-v3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    low_cpu_mem_usage=True
    )

Sprawdzenie działania modelu. Wykorzystamy potok przetwarzania `transformers.pipeline("text-generation", ...)` zaimplementowany w HuggingFace aby zautomatyzować generowanie dłuższych sekwencji tekstowywch, bez konieczności iteracyjnego samplowania kolejnych tokenów.

In [ ]:
import transformers

pipeline = transformers.pipeline("text-generation", model=model, tokenizer=tokenizer)

text = "Jeśli nie chcesz mojej zgudy, krokodyla"

sequences = pipeline(max_new_tokens=100, do_sample=True, top_k=10, eos_token_id=tokenizer.eos_token_id, text_inputs=text)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

In [ ]:
text = "W sławnym mieście Pacanowie"

sequences = pipeline(max_new_tokens=100, do_sample=True, top_k=10, eos_token_id=tokenizer.eos_token_id, text_inputs=text)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Modele z rodziny Bielik wersja 3 wykorzystują własny tokenizator APT4 zoptymalizowany do języka polskiego.

In [ ]:
print(tokenizer)

In [ ]:
txt = "Litwo, Ojczyzno moja! ty jesteś jak zdrowie; Ile cię trzeba cenić,"

In [ ]:
print(txt)
print()

tokenized_txt = tokenizer.tokenize(txt)
print("Tekst po tokenizacji:")
print(tokenizer.tokenize(txt))
print()

print("Tekst po zamianie na identyfikatory tokenów:")
encoded_txt = tokenizer.convert_tokens_to_ids(tokenized_txt)
print(encoded_txt)
print()

print("Zrekonstruowany tekst:")
decoded_txt = tokenizer.decode(encoded_txt)
print(decoded_txt)
print()


In [ ]:
print(f"Rozmiar słownika: {tokenizer.vocab_size} tokenów")

selected_token_ids = random.sample(range(tokenizer.vocab_size), k=20)
for token_id in selected_token_ids:
    print(f"id: {token_id} -> {tokenizer.decode(token_id)}")